# IF participant-account distribution
Compare `INSTRUMENTO_FINANCEIRO.NUM_CONTA_PARTICIPANTE` in synthetic outputs against `onprem-export-full`. Upload this notebook and `compare_if_account_distribution.py` together to OCI Data Science, using Python 3.11 and your existing OCI-enabled `spark` session.

Defaults target the five products in run `20260911T221522Z-a2ef1130`; edit paths/products below for another run. Source and synthetic data are read-only. CSV output is disabled until you set `REPORT_URI`. No production results are embedded in this notebook.

In [ ]:
RUN_ID = '20260911T221522Z-a2ef1130'
EXPORT_BASE = 'oci://oci-st-blc-engordai-qab-n@gr97zovfhcmu/onprem-export-full'
RUN_BASE = f'oci://oci-st-blc-engordai-qab-n@gr97zovfhcmu/pipeline-runs/portuguesa/{RUN_ID}'
PRODUCT_TYPES = {
    'cdb_simplificado': 49,
    'cdb_resgate': 49,
    'cdb_escalonamento': 49,
    'rdb_resgate': 50,
    'rdb_inclusao': 50,
}
PRODUCTS = list(PRODUCT_TYPES)  # Select fewer products here if desired.
SYNTHETIC_BASES = {p: f'{RUN_BASE}/products/{p}/synthetic' for p in PRODUCTS}
HELPER_FILE = 'compare_if_account_distribution.py'
INCLUDE_CLONE_MAP = True
BASELINE_TO_VIEW = 'full_export'
TOP_N = 30
REPORT_URI = None  # Optional NEW directory outside all source/synthetic input trees.

## Read the baselines correctly
| Baseline | Meaning |
| --- | --- |
| `full_export` | Every source IF, across all types and exclusion states. Context, not a like-for-like CDB/RDB sample. |
| `same_type` | Source IFs with the product's `NUM_TIPO_IF` (49 CDB, 50 RDB), including excluded rows. |
| `active_same_type` | Same type, with `DAT_EXCLUSAO IS NULL`. Still not the exact scenario eligibility pool. |
| `selected_sources` | Distinct original IFs referenced by the clone map. Separates selection from replication. |
| `clone_weighted` | Original accounts repeated once per mapped clone. Expected synthetic distribution if accounts were preserved. |

`DELTA_PP = SYNTHETIC_PCT - SOURCE_PCT`: positive means greater representation in synthetic output. Percentages use each cohort's own IF total, not row totals from joined child tables. NULL/blank accounts remain an explicit NULL bucket. Empty populations produce undefined (NULL) percentages, not a false zero difference.

`TOTAL_VARIATION_PCT` is half the sum of absolute percentage-point differences: 0 means identical marginal distributions, 100 means disjoint ones. `CHANGED_ACCOUNT_IFS` checks each mapped IF, so even account swaps that cancel out in the marginal distribution are detected.

Duplicate IF IDs, incomplete maps, and wrong synthetic IF types fail rather than silently distort counts. If maps are intentionally unavailable, set `INCLUDE_CLONE_MAP=False`; the last two baselines and change checks will be unavailable, not reported as passing. No date window is inferred from the rewritten synthetic operational dates.

In [ ]:
from functools import reduce
from pathlib import Path
import runpy
from pyspark import StorageLevel
from pyspark.sql import functions as F

if 'spark' not in globals():
    raise RuntimeError('Use an existing Spark session configured for OCI Object Storage.')
if not PRODUCTS or len(PRODUCTS) != len(set(PRODUCTS)):
    raise ValueError('Select at least one product, without duplicates.')
if any(p not in PRODUCT_TYPES or p not in SYNTHETIC_BASES for p in PRODUCTS):
    raise ValueError('Every selected product needs an IF type and synthetic base URI.')
if not Path(HELPER_FILE).is_file():
    raise FileNotFoundError(f'Upload the comparison helper or adjust HELPER_FILE: {HELPER_FILE}')
compare_if_accounts = runpy.run_path(HELPER_FILE)['compare_if_accounts']

# Release only caches created by a previous execution of this notebook.
for frame in globals().get('if_account_caches', []):
    frame.unpersist()
if_account_caches = []
if_account_reports = {}
source_if = spark.read.parquet(EXPORT_BASE.rstrip('/') + '/INSTRUMENTO_FINANCEIRO').select(
    'NUM_IF', 'NUM_CONTA_PARTICIPANTE', 'NUM_TIPO_IF', 'DAT_EXCLUSAO'
).persist(StorageLevel.MEMORY_AND_DISK)
if_account_caches.append(source_if)
print(f'Source: {EXPORT_BASE}; run: {RUN_ID}; products: {PRODUCTS}')

In [ ]:
for product in PRODUCTS:
    base = SYNTHETIC_BASES[product].rstrip('/')
    print(f'Comparing {product}: {base}')
    synthetic_if = spark.read.parquet(f'{base}/INSTRUMENTO_FINANCEIRO').select(
        'NUM_IF', 'NUM_CONTA_PARTICIPANTE', 'NUM_TIPO_IF'
    ).persist(StorageLevel.MEMORY_AND_DISK)
    if_account_caches.append(synthetic_if)
    clone_map = None
    if INCLUDE_CLONE_MAP:
        clone_map = spark.read.parquet(f'{base}/MAPA_CLONE_NUM_IF').select(
            'NUM_IF_ORIG', 'K', 'NUM_IF_NOVO'
        ).persist(StorageLevel.MEMORY_AND_DISK)
        if_account_caches.append(clone_map)
    report = compare_if_accounts(source_if, synthetic_if, clone_map, PRODUCT_TYPES[product])
    for frame in report.values():
        if frame is not None:
            frame.persist(StorageLevel.MEMORY_AND_DISK)
            if_account_caches.append(frame)
    if_account_reports[product] = report
    report['summary'].orderBy('BASELINE').show(10, truncate=False)

def combine_reports(name):
    frames = [
        report[name].withColumn('PRODUCT', F.lit(product)).withColumn('RUN_ID', F.lit(RUN_ID))
        for product, report in if_account_reports.items() if report[name] is not None
    ]
    return reduce(lambda a, b: a.unionByName(b), frames) if frames else None

account_distribution = combine_reports('distribution')
account_summary = combine_reports('summary')
account_changes = combine_reports('account_changes')

## Largest account-share differences
The default is the requested `full_export` comparison. Change `BASELINE_TO_VIEW` to `active_same_type` for active CDB/RDB source populations, or to `clone_weighted` to isolate whether cloning preserved accounts. Only display values are rounded; report values retain full precision. All accounts, including source-only and synthetic-only accounts, remain in `account_distribution` and the optional CSV export.

In [ ]:
allowed_baselines = {'full_export', 'same_type', 'active_same_type'}
if INCLUDE_CLONE_MAP:
    allowed_baselines |= {'selected_sources', 'clone_weighted'}
if BASELINE_TO_VIEW not in allowed_baselines:
    raise ValueError(f'Unavailable baseline: {BASELINE_TO_VIEW}')
for product in PRODUCTS:
    print(f'{product}: {BASELINE_TO_VIEW}, top {TOP_N} by absolute percentage-point change')
    comparison = if_account_reports[product]['distribution'].where(
        F.col('BASELINE') == BASELINE_TO_VIEW
    ).orderBy(F.abs(F.col('DELTA_PP')).desc_nulls_last(), 'NUM_CONTA_PARTICIPANTE')
    comparison.select(
        'NUM_CONTA_PARTICIPANTE', 'SOURCE_IF_COUNT', 'SYNTHETIC_IF_COUNT',
        F.round('SOURCE_PCT', 6).alias('SOURCE_PCT'),
        F.round('SYNTHETIC_PCT', 6).alias('SYNTHETIC_PCT'),
        F.round('DELTA_PP', 6).alias('DELTA_PP'), 'PRESENCE'
    ).show(TOP_N, truncate=False)

if account_changes is None:
    print('Per-IF preservation check unavailable: clone maps were disabled.')
else:
    print('Mapped IFs whose participant account changed (bounded preview):')
    account_changes.show(TOP_N, truncate=False)

In [ ]:
if REPORT_URI is None:
    print('CSV export disabled. Set REPORT_URI to a new report directory to export all rows.')
else:
    report_base = REPORT_URI.rstrip('/')
    inputs = [EXPORT_BASE.rstrip('/')] + [SYNTHETIC_BASES[p].rstrip('/') for p in PRODUCTS]
    if not report_base or any(
        report_base == path or report_base.startswith(path + '/') or path.startswith(report_base + '/')
        for path in inputs
    ):
        raise ValueError('REPORT_URI must be separate from every input tree.')
    for name, frame in (
        ('distribution', account_distribution), ('summary', account_summary),
        ('account_changes', account_changes),
    ):
        if frame is not None:
            frame.write.mode('errorifexists').option('header', True).option(
                'nullValue', '<NULL>'
            ).csv(f'{report_base}/{name}')
    print(f'Wrote distributed CSV directories under {report_base}; existing reports are never overwritten.')

## Release notebook caches
Run this after inspecting/exporting the results. The helper collects only scalar integrity counts; Spark performs account grouping and joins in the cluster. No full IF dataset is converted to pandas or collected into Python. Keep inputs unchanged while this notebook runs.

In [ ]:
for frame in if_account_caches:
    frame.unpersist()
if_account_caches = []
print('Comparison caches released; the shared Spark session is still running.')